In [1]:
# Check whether easydiffraction is installed; install it if needed.
# Required for remote environments such as Google Colab.
import importlib.util

if importlib.util.find_spec('easydiffraction') is None:
    %pip install easydiffraction

# Structure Refinement: YAlO3+Al2O3, SPODI

This example demonstrates a staged two-phase Rietveld refinement of
yttrium aluminium perovskite YAlO3 (or YAP) with a small Al2O3
impurity using constant wavelength neutron powder diffraction data
measured at 3 K on SPODI at MLZ.

The workflow defines both structures, configures the experiment, and
refines the cell, scale, profile, background, and atom parameters of
both phases in stages.

## 🛠️ Import Library

In [2]:
import easydiffraction as edi

## 📦 Define Project

### Create Project

In [3]:
project = edi.Project(
    name='yap_3k',
    description='Two-phase YAlO3 and Al2O3 refinement using 3 K data from SPODI at MLZ.',
)

### Save Initial Project

In [4]:
project.save_as(dir_path='projects/refine-yap-3k')

Saving project 📦 'yap_3k' to '../../../projects/refine-yap-3k'


├── 📄 project.edi
├── 📁 structures/
├── 📁 experiments/
├── 📁 analysis/
│   └── 📄 analysis.edi
└── 📁 reports/
    └── 📄 yap_3k.html


## 🧩 Define Structures

### Create Structure 1: YAlO3

Preserve the orthorhombic Pbnm setting used in FullProf. In
EasyDiffraction this is represented by the standard space-group
symbol `P n m a` with coordinate-system code `cab`. The cell axes and
atom coordinates below therefore stay in the original Pbnm setting.

FullProf's PCR occupancies include the site multiplicity divided by
the general-position multiplicity. Here each atom site is fully
occupied: the PCR values 0.5 for Y, Al, and O1, and 1.0 for O2, all
become an occupancy of 1.0. Displacement parameters are entered as
Biso, matching the PCR file.

In [5]:
yap_cif = """
data_yap

_cell.length_a 5.18
_cell.length_b 5.33
_cell.length_c 7.37
_cell.angle_alpha 90.
_cell.angle_beta 90.
_cell.angle_gamma 90.

_space_group.name_h_m "P n m a"
_space_group.coord_system_code cab

loop_
_atom_site.id
_atom_site.type_symbol
_atom_site.fract_x
_atom_site.fract_y
_atom_site.fract_z
_atom_site.occupancy
_atom_site.adp_iso
_atom_site.adp_type
Y  Y   0.0100  0.5500 0.2500 1.0 0.12 Biso
Al Al  0.0000  0.0000 0.0000 1.0 0.13 Biso
O1 O  -0.0800 -0.0200 0.2500 1.0 0.06 Biso
O2 O   0.2000  0.2900 0.0400 1.0 0.14 Biso
"""

In [6]:
project.structures.add_from_cif_str(yap_cif)

In [7]:
yap = project.structures['yap']

### Display Structure 1: YAlO3

In [8]:
yap.show_as_text()

Structure 🧩 'yap' as text


,Edi
1,data_yap
2,
3,_cell.length_a 5.18
4,_cell.length_b 5.33
5,_cell.length_c 7.37
6,_cell.angle_alpha 90.
7,_cell.angle_beta 90.
8,_cell.angle_gamma 90.
9,
10,"_space_group.name_h_m ""P n m a"""


In [9]:
project.display.structure(struct_name='yap')

Structure 🧩 'yap' (Atom view type: 'covalent')


### Create Structure 2: Al2O3

Define the corundum impurity in the hexagonal setting of R-3c. The
FullProf PCR occupancies of 2/3 for Al and 1 for O also describe fully
occupied sites. Refine its two independent cell lengths, Al z and O x
coordinates, and both Biso values, as specified by the PCR codewords.
The PCR contains a negative Al Biso. EasyDiffraction requires a
nonnegative input value, so start this parameter at 0.1 Å² and refine
it alongside O Biso.

In [10]:
alumina = edi.StructureFactory.from_scratch(name='alumina')

#### Set Space Group

In [11]:
alumina.space_group.name_h_m = 'R -3 c'
alumina.space_group.coord_system_code = 'h'

#### Set Unit Cell

In [12]:
alumina.cell.length_a = 4.75
alumina.cell.length_c = 12.95

#### Set Atom Sites

In [13]:
alumina.atom_sites.create(
    id='Al1',
    type_symbol='Al',
    fract_x=0.0,
    fract_y=0.0,
    fract_z=0.33351,
    occupancy=1.0,
    adp_type='Biso',
    adp_iso=0.1,
)
alumina.atom_sites.create(
    id='O1',
    type_symbol='O',
    fract_x=0.3503,
    fract_y=0.0,
    fract_z=0.25,
    occupancy=1.0,
    adp_type='Biso',
    adp_iso=1.22884,
)

In [14]:
project.structures.add(alumina)

### Display Structure 2: Al2O3

In [15]:
alumina.show_as_text()

Structure 🧩 'alumina' as text


,Edi
1,data_alumina
2,
3,_cell.length_a 4.75
4,_cell.length_b 4.75
5,_cell.length_c 12.95
6,_cell.angle_alpha 90.
7,_cell.angle_beta 90.
8,_cell.angle_gamma 120.
9,
10,"_space_group.name_h_m ""R -3 c"""


In [16]:
project.display.structure(struct_name='alumina')

Structure 🧩 'alumina' (Atom view type: 'covalent')


## 🔬 Define Experiment

### Download Measured Data

Download the YAlO3 + Al2O3 pattern from the EasyDiffraction online
data repository. The three columns contain 2-theta in degrees,
intensity, and its standard uncertainty. They are copied from the
original SPODI dataset without changing the measured values.

In [17]:
data_path = edi.download_data('meas-yap-spodi', destination='data')

Getting data...


Data 'meas-yap-spodi': YAlO3 + Al2O3, SPODI (MLZ), 3 K


✅ Data 'meas-yap-spodi' downloaded to '../../../data/meas-yap-spodi.xye'


In [18]:
project.experiments.add_from_data_path(
    name='yap_3k',
    data_path=data_path,
    sample_form='powder',
    beam_mode='constant wavelength',
    radiation_probe='neutron',
)

Data loaded successfully


Experiment 🔬 'yap_3k'. Number of data points: 3000.


In [19]:
expt = project.experiments['yap_3k']

### Set Instrument

Use the neutron wavelength reported for the SPODI dataset and start
with zero 2-theta offset.

In [20]:
expt.instrument.setup_wavelength = 1.54816
expt.instrument.calib_twotheta_offset = 0.0

### Set Peak Profile

Use a pseudo-Voigt profile with Bérar-Baldinozzi asymmetry.

In [21]:
expt.peak.show_supported()

Peak types


,,Type,Description
1,*,pseudo-voigt,CWL pseudo-Voigt profile
2,,pseudo-voigt + berar-baldinozzi asymmetry,CWL pseudo-Voigt profile with Berar-Baldinozzi asymmetry correction.


In [22]:
expt.peak.type = 'pseudo-voigt + berar-baldinozzi asymmetry'

⚠️ Switching peak profile type adds these settings with defaults:
• asym_beba_a0=0.0
• asym_beba_a1=0.0
• asym_beba_b0=0.0
• asym_beba_b1=0.0


Peak profile type for experiment 'yap_3k' changed to


pseudo-voigt + berar-baldinozzi asymmetry


In [23]:
expt.peak.broad_gauss_u = 0.04
expt.peak.broad_gauss_v = -0.05
expt.peak.broad_gauss_w = 0.10
expt.peak.broad_lorentz_x = 0.0
expt.peak.broad_lorentz_y = 0.01

In [24]:
expt.peak.asym_beba_a0 = 0.0
expt.peak.asym_beba_b0 = 0.0
expt.peak.asym_beba_a1 = 0.0
expt.peak.asym_beba_b1 = 0.0

In [25]:
expt.peak.cutoff_fwhm = 8.0

### Set Absorption

Apply the cylindrical-sample Hewat correction with the absorption
radius product from FullProf.

In [26]:
expt.absorption.type = 'cylinder-hewat'
expt.absorption.mu_r = 0.0221

Absorption type changed to


cylinder-hewat


### Set Excluded Regions

In [27]:
expt.excluded_regions.create(id='1', start=0.0, end=4.0)
expt.excluded_regions.create(id='2', start=153.95, end=180.0)

### Set Background

Estimate the initial line-segment background from the measured pattern.
This first estimate does not use a calculated structural model.

In [28]:
expt.background.auto_estimate(use_model=False)

In [29]:
expt.background.show()

Line-segment background points


,Position,Intensity
1,4.05000,1303.00000
2,5.95000,2193.97204
3,18.20000,1997.00000
4,37.35000,2147.12282
5,47.70000,2092.00000
6,59.25000,1977.00000
7,151.95000,3468.48635


### Set Linked Structures

Give each phase an independent scale factor. Scale factors are fitted
intensity multipliers, rather than phase weight fractions.

In [30]:
expt.linked_structures.create(structure_id='yap', scale=30)
expt.linked_structures.create(structure_id='alumina', scale=0.1)

In [31]:
expt.show_as_text()

Experiment 🔬 'yap_3k' as text


,Edi
1,data_yap_3k
2,
3,_experiment_type.sample_form powder
4,"_experiment_type.beam_mode ""constant wavelength"""
5,_experiment_type.radiation_probe neutron
6,_experiment_type.scattering_type bragg
7,
8,_diffrn.ambient_temperature ?
9,_diffrn.ambient_pressure ?
10,_diffrn.ambient_magnetic_field ?


## 🚀 Perform Analysis

### Display Initial Pattern

In [32]:
project.display.pattern(expt_name='yap_3k')

In [33]:
project.display.pattern(expt_name='yap_3k', x_min=134, x_max=146)

### Select Calculator

In [34]:
expt.calculator.show_supported()

Calculator types


,,Type,Description
1,,crysfml,CrysFML library for crystallographic calculations
2,*,cryspy,CrysPy library for crystallographic calculations


In [35]:
expt.calculator.type = 'cryspy'

Calculator for experiment 'yap_3k' already set to


cryspy


### Select Minimizer

In [36]:
project.analysis.minimizer.show_supported()

Minimizer types


,,Type,Description
1,,bumps,BUMPS library using the default Levenberg-Marquardt method
2,,bumps (amoeba),BUMPS library with Nelder-Mead simplex method
3,,bumps (de),BUMPS library with differential evolution method
4,,bumps (dream),BUMPS library with DREAM Bayesian sampling
5,,bumps (lm),BUMPS library with Levenberg-Marquardt method
6,,dfols,DFO-LS library for derivative-free least-squares optimization
7,,emcee,emcee affine-invariant ensemble Bayesian sampling
8,,lmfit,LMFIT library using the default Levenberg-Marquardt method
9,,lmfit (least_squares),LMFIT library with SciPy's trust region reflective algorithm
10,*,lmfit (leastsq),LMFIT library with Levenberg-Marquardt least squares method


In [37]:
project.analysis.minimizer.type = 'bumps (lm)'

⚠️ Switching minimizer type removes these settings:
• gradient_tolerance


Current minimizer changed to


bumps (lm)


In [38]:
project.analysis.minimizer.max_iterations = 500
project.analysis.minimizer.chi_square_change_tolerance = 1e-2

### Perform Fit 1/3: Cell, Scale, and Background

First refine the independent cell lengths of both phases, both phase
scales, the instrument zero offset, and the automatically estimated
background intensities. Hexagonal symmetry couples the Al2O3 b length
to a, leaving only a and c independent.

In [39]:
yap.cell.length_a.free = True
yap.cell.length_b.free = True
yap.cell.length_c.free = True

alumina.cell.length_a.free = True
alumina.cell.length_c.free = True

expt.linked_structures['yap'].scale.free = True
expt.linked_structures['alumina'].scale.free = True

expt.instrument.calib_twotheta_offset.free = True

for point in expt.background:
    point.intensity.free = True

In [40]:
project.display.parameters.free()

Free parameters for both structures (🧩 data blocks) and experiments (🔬 data blocks)


,datablock,category,entry,parameter,value,uncertainty,min,max,units
1,yap,cell,,length_a,5.18000,,-inf,inf,Å
2,yap,cell,,length_b,5.33000,,-inf,inf,Å
3,yap,cell,,length_c,7.37000,,-inf,inf,Å
4,alumina,cell,,length_a,4.75000,,-inf,inf,Å
5,alumina,cell,,length_c,12.95000,,-inf,inf,Å
6,yap_3k,linked_structure,yap,scale,30.00000,,-inf,inf,
7,yap_3k,linked_structure,alumina,scale,0.10000,,-inf,inf,
8,yap_3k,instrument,,twotheta_offset,0.00000,,-inf,inf,deg
9,yap_3k,background,1,intensity,1303.00000,,-inf,inf,
10,yap_3k,background,2,intensity,2193.97204,,-inf,inf,


In [41]:
project.analysis.fit()

<IPython.core.display.Javascript object>

Standard fitting


📋 Using experiment 🔬 'yap_3k' for 'single' fitting


🚀 Starting fit process with 'bumps (lm)'...


📈 Goodness-of-fit progress:


,iteration,time (s),χ²,change / status
1,1,0.29,1170.67,
2,17,2.45,266.00,77.3% ↓
3,33,4.66,74.80,71.9% ↓
4,49,7.06,66.39,11.3% ↓
5,67,17.70,66.03,


🏆 Best goodness-of-fit (reduced χ²) is 66.03 at iteration 67


✅ Fitting complete.


In [42]:
project.display.fit.results()

⚙️ Settings used:


,Name,Value,Description
1,max_iterations,500,Maximum solver iterations.
2,chi_square_change_tolerance,0.01,Relative change in the objective (chi-square) used to stop fitting.
3,parameter_change_tolerance,1e-08,Relative change in fitted parameters used to stop fitting.


📋 Least-squares fit results:


,Metric,Value
1,🧪 Minimizer,bumps (lm)
2,✅ Overall status,success
3,⏱️ Fitting time (seconds),17.70
4,📏 Goodness-of-fit (reduced χ²),66.03
5,"📏 R-factor (Rf, %)",8.19
6,"📏 R-factor squared (Rf², %)",10.54
7,"📏 Weighted R-factor (wR, %)",10.63


📈 Refined parameters:


,datablock,category,entry,parameter,units,start,value,s.u.,change
1,yap,cell,,length_a,Å,5.1800,5.1736,0.0001,0.12 % ↓
2,yap,cell,,length_b,Å,5.3300,5.3278,0.0001,0.04 % ↓
3,yap,cell,,length_c,Å,7.3700,7.3624,0.0001,0.10 % ↓
4,alumina,cell,,length_a,Å,4.7500,4.7556,0.0012,0.12 % ↑
5,alumina,cell,,length_c,Å,12.9500,12.9893,0.0063,0.30 % ↑
6,yap_3k,linked_structure,yap,scale,,30.0000,27.0708,0.0851,9.76 % ↓
7,yap_3k,linked_structure,alumina,scale,,0.1000,0.1940,0.0166,93.98 % ↑
8,yap_3k,instrument,,twotheta_offset,deg,0.0000,0.0044,0.0010,N/A
9,yap_3k,background,1,intensity,,1303.0000,1364.5791,89.8261,4.73 % ↑
10,yap_3k,background,2,intensity,,2193.9720,2366.3256,42.1391,7.86 % ↑


### Perform Fit 2/3: Peak Profile

Add the Gaussian and Lorentzian broadening and asymmetry parameters
to the refinement. The background intensities remain free.

In [43]:
expt.peak.broad_gauss_u.free = True
expt.peak.broad_gauss_v.free = True
expt.peak.broad_gauss_w.free = True
expt.peak.broad_lorentz_y.free = True

expt.peak.asym_beba_a0.free = True
# expt.peak.asym_beba_b0.free = True
# expt.peak.asym_beba_a1.free = True
expt.peak.asym_beba_b1.free = True

In [44]:
project.display.parameters.free()

Free parameters for both structures (🧩 data blocks) and experiments (🔬 data blocks)


,datablock,category,entry,parameter,value,uncertainty,min,max,units
1,yap,cell,,length_a,5.17364,0.00007,-inf,inf,Å
2,yap,cell,,length_b,5.32778,0.00007,-inf,inf,Å
3,yap,cell,,length_c,7.36239,0.00010,-inf,inf,Å
4,alumina,cell,,length_a,4.75560,0.00119,-inf,inf,Å
5,alumina,cell,,length_c,12.98925,0.00626,-inf,inf,Å
6,yap_3k,linked_structure,yap,scale,27.07084,0.08509,-inf,inf,
7,yap_3k,linked_structure,alumina,scale,0.19398,0.01660,-inf,inf,
8,yap_3k,peak,,asym_beba_a0,0.00000,,-inf,inf,
9,yap_3k,peak,,asym_beba_b1,0.00000,,-inf,inf,
10,yap_3k,peak,,broad_gauss_u,0.04000,,-inf,inf,deg²


In [45]:
project.analysis.fit()

<IPython.core.display.Javascript object>

Standard fitting


📋 Using experiment 🔬 'yap_3k' for 'single' fitting


🚀 Starting fit process with 'bumps (lm)'...


📈 Goodness-of-fit progress:


,iteration,time (s),χ²,change / status
1,1,0.19,66.17,
2,23,5.07,59.24,10.5% ↓
3,45,10.06,58.60,1.1% ↓
4,69,25.78,58.50,


🏆 Best goodness-of-fit (reduced χ²) is 58.50 at iteration 69


✅ Fitting complete.


In [46]:
project.display.fit.results()

⚙️ Settings used:


,Name,Value,Description
1,max_iterations,500,Maximum solver iterations.
2,chi_square_change_tolerance,0.01,Relative change in the objective (chi-square) used to stop fitting.
3,parameter_change_tolerance,1e-08,Relative change in fitted parameters used to stop fitting.


📋 Least-squares fit results:


,Metric,Value
1,🧪 Minimizer,bumps (lm)
2,✅ Overall status,success
3,⏱️ Fitting time (seconds),25.78
4,📏 Goodness-of-fit (reduced χ²),58.50
5,"📏 R-factor (Rf, %)",7.66
6,"📏 R-factor squared (Rf², %)",10.03
7,"📏 Weighted R-factor (wR, %)",9.99


📈 Refined parameters:


,datablock,category,entry,parameter,units,start,value,s.u.,change
1,yap,cell,,length_a,Å,5.1736,5.1728,0.0001,0.02 % ↓
2,yap,cell,,length_b,Å,5.3278,5.3270,0.0001,0.01 % ↓
3,yap,cell,,length_c,Å,7.3624,7.3610,0.0002,0.02 % ↓
4,alumina,cell,,length_a,Å,4.7556,4.7577,0.0011,0.04 % ↑
5,alumina,cell,,length_c,Å,12.9893,12.9761,0.0059,0.10 % ↓
6,yap_3k,linked_structure,yap,scale,,27.0708,28.0693,0.1137,3.69 % ↑
7,yap_3k,linked_structure,alumina,scale,,0.1940,0.1941,0.0161,0.07 % ↑
8,yap_3k,peak,,asym_beba_a0,,0.0000,-0.1818,0.0152,N/A
9,yap_3k,peak,,asym_beba_b1,,0.0000,-0.0612,0.0050,N/A
10,yap_3k,peak,,broad_gauss_u,deg²,0.0400,0.0364,0.0020,8.95 % ↓


### Perform Fit 3/3: Model-Guided Background and Atom Parameters

Replace the initial background with a new estimate based on the fitted
peak model. Automatically generated points are fixed by default, so
mark their intensities free before fitting them with the atom parameters.

In [47]:
expt.background.auto_estimate(use_model=True)

In [48]:
expt.background.show()

Line-segment background points


,Position,Intensity
1,4.05000,1303.00000
2,5.85000,2173.00000
3,7.30000,2414.80830
4,17.85000,2010.28100
5,24.75000,2041.00000
6,35.00000,1968.46433
7,39.35000,1606.73948
8,41.45000,0.00000
9,43.95000,1695.97793
10,44.95000,2367.00000


In [49]:
for point in expt.background:
    point.intensity.free = True

Refine the independent Y and O coordinates in Pbnm and the
isotropic displacement parameters of both phases. Symmetry keeps Y
and O1 in YAlO3 at z = 1/4 and Al at the origin. For Al2O3, refine
Al1 z and O1 x. Occupancies remain fixed
at 1.0, and all coordinates fixed by symmetry remain fixed.

In [50]:
yap.atom_sites['Y'].fract_x.free = True
yap.atom_sites['Y'].fract_y.free = True
yap.atom_sites['O1'].fract_x.free = True
yap.atom_sites['O1'].fract_y.free = True
yap.atom_sites['O2'].fract_x.free = True
yap.atom_sites['O2'].fract_y.free = True
yap.atom_sites['O2'].fract_z.free = True

alumina.atom_sites['Al1'].fract_z.free = True
alumina.atom_sites['O1'].fract_x.free = True

for structure in (yap, alumina):
    for atom in structure.atom_sites:
        atom.adp_iso.free = True

In [51]:
project.display.parameters.free()

Free parameters for both structures (🧩 data blocks) and experiments (🔬 data blocks)


,datablock,category,entry,parameter,value,uncertainty,min,max,units
1,yap,cell,,length_a,5.17278,0.00012,-inf,inf,Å
2,yap,cell,,length_b,5.32698,0.00012,-inf,inf,Å
3,yap,cell,,length_c,7.36105,0.00017,-inf,inf,Å
4,yap,atom_site,Y,fract_x,0.01000,,-inf,inf,
5,yap,atom_site,Y,fract_y,0.55000,,-inf,inf,
6,yap,atom_site,Y,adp_iso,0.12000,,-inf,inf,Å²
7,yap,atom_site,Al,adp_iso,0.13000,,-inf,inf,Å²
8,yap,atom_site,O1,fract_x,-0.08000,,-inf,inf,
9,yap,atom_site,O1,fract_y,-0.02000,,-inf,inf,
10,yap,atom_site,O1,adp_iso,0.06000,,-inf,inf,Å²


In [52]:
project.analysis.fit()

<IPython.core.display.Javascript object>

Standard fitting


📋 Using experiment 🔬 'yap_3k' for 'single' fitting


🚀 Starting fit process with 'bumps (lm)'...


📈 Goodness-of-fit progress:


,iteration,time (s),χ²,change / status
1,1,0.13,78.60,
2,51,8.41,9.93,87.4% ↓
3,101,17.18,9.28,6.6% ↓
4,153,41.64,9.25,


🏆 Best goodness-of-fit (reduced χ²) is 9.25 at iteration 153


✅ Fitting complete.


### Inspect Results

Review the fit statistics, refined parameters, and correlations.
Inspect the full pattern and a closer view with impurity reflections.

In [53]:
project.display.fit.results()

⚙️ Settings used:


,Name,Value,Description
1,max_iterations,500,Maximum solver iterations.
2,chi_square_change_tolerance,0.01,Relative change in the objective (chi-square) used to stop fitting.
3,parameter_change_tolerance,1e-08,Relative change in fitted parameters used to stop fitting.


📋 Least-squares fit results:


,Metric,Value
1,🧪 Minimizer,bumps (lm)
2,✅ Overall status,success
3,⏱️ Fitting time (seconds),41.64
4,📏 Goodness-of-fit (reduced χ²),9.25
5,"📏 R-factor (Rf, %)",3.13
6,"📏 R-factor squared (Rf², %)",3.71
7,"📏 Weighted R-factor (wR, %)",3.96


📈 Refined parameters:


,datablock,category,entry,parameter,units,start,value,s.u.,change
1,yap,cell,,length_a,Å,5.1728,5.1728,0.0000,0.00 % ↑
2,yap,cell,,length_b,Å,5.3270,5.3271,0.0000,0.00 % ↑
3,yap,cell,,length_c,Å,7.3610,7.3613,0.0001,0.00 % ↑
4,yap,atom_site,Y,fract_x,,0.0100,0.0123,0.0001,22.93 % ↑
5,yap,atom_site,Y,fract_y,,0.5500,0.5541,0.0001,0.74 % ↑
6,yap,atom_site,Y,adp_iso,Å²,0.1200,0.0711,0.0086,40.77 % ↓
7,yap,atom_site,Al,adp_iso,Å²,0.1300,0.0938,0.0149,27.81 % ↓
8,yap,atom_site,O1,fract_x,,-0.0800,-0.0839,0.0001,4.91 % ↑
9,yap,atom_site,O1,fract_y,,-0.0200,-0.0219,0.0001,9.38 % ↑
10,yap,atom_site,O1,adp_iso,Å²,0.0600,0.0280,0.0096,53.41 % ↓


In [54]:
project.display.fit.correlations()

In [55]:
project.display.pattern(expt_name='yap_3k')

In [56]:
project.display.pattern(expt_name='yap_3k', x_min=134, x_max=146)

## 💾 Save Project

Save the refined model and analysis results in the project directory.

In [57]:
project.save()

Saving project 📦 'yap_3k' to '../../../projects/refine-yap-3k'


├── 📄 project.edi
├── 📁 structures/
│   └── 📄 yap.edi
│   └── 📄 alumina.edi
├── 📁 experiments/
│   └── 📄 yap_3k.edi
├── 📁 analysis/
│   └── 📄 analysis.edi
└── 📁 reports/
    └── 📄 yap_3k.html
